# Imports

In [2]:
# Linear algebra
import numpy as np

# Dataframes
import pandas as pd

# Path
import os
data_path = os.path.join('..', 'data')

# Plotting
# import seaborn as sns
import matplotlib.pyplot as plt

# Pandas faster apply
import swifter

# Progress bar
from tqdm import tqdm

C:\Users\wsega\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Data

In [27]:
# Load test data
test_with_results = pd.read_pickle(os.path.join(data_path, '2_test_with_bart30.pkl'))

# Load tRNA abundancies
tRNA_abundancies = pd.read_csv(os.path.join(data_path, 'SDA', 'tRNA_TCGA_Healthy', 'TCGAhealthy_ABS.csv'))
tRNA_abundancies.set_index('Unnamed: 0', inplace=True)
tRNA_abundancies.index.rename('anticodon', inplace=True)
# SDAw_TCGA = pd.read_csv(os.path.join(data_path, 'SDA', 'tRNA_TCGA_Healthy', 'TCGAhealthy_ABS.csv'))

# Organize df to (codons x samples) shape
# SDAw_TCGA.rename(columns={'Unnamed: 0': 'samples'}, inplace=True)
# new_column_names = SDAw_TCGA['samples']
# SDAw_TCGA = SDAw_TCGA.drop(columns=['samples']).T.rename(new_column_names, axis=1)
# SDAw_TCGA.index.rename('Codon', inplace=True)
# # Remove first 3 letters (amino acid) from index entries
# SDAw_TCGA.index = SDAw_TCGA.index.str[3:]
# # Drop NaN samples
# SDAw_TCGA.dropna(axis=1, inplace=True)


# Calculate Relative tRNA Adaptation Index (RtAI)

In [22]:
# calculate sum of each column up to the last one (not included) minus the last one
tRNA_abundancies.loc['sum'] = tRNA_abundancies.iloc[:-1, :].sum(axis=0) #- tRNA_abundancies.iloc[-1, :]

In [23]:
tRNA_abundancies

,COAD-TCGA-A6-2671-11A-01R-1757-13_mirna,COAD-TCGA-A6-2680-11A-01R-1757-13_mirna,COAD-TCGA-AA-3525-11A-01R-1757-13_mirna,READ-TCGA-AF-2689-11A-01R-1757-13_mirna,COAD-TCGA-A6-2685-11A-01R-1757-13_mirna,COAD-TCGA-A6-2683-11A-01R-1757-13_mirna,COAD-TCGA-AA-3527-11A-01R-1757-13_mirna,COAD-TCGA-A6-2684-11A-01R-1757-13_mirna,READ-TCGA-AF-2691-11A-01R-1757-13_mirna,GBM-TCGA-06-0675-11A-32R-A36C-13_mirna,...,LUAD-TCGA-44-6144-11A-01H-2169-13_mirnaB,BLCA-TCGA-BL-A13J-11A-13R-A10V-13_mirnaB,BRCA-TCGA-BH-A0E1-11A-13R-A090-13_mirnaA,UCEC-TCGA-DI-A2QU-11A-11R-A18L-13_mirnaB,STAD-TCGA-HU-A4GH-11A-11R-A360-13_mirna,PAAD-TCGA-HV-A5A3-11A-11R-A26Y-13_mirna,UCEC-TCGA-FL-A1YQ-11A-11R-A17A-13_mirnaA,LUSC-TCGA-56-7580-11A-01H-2044-13_mirnaA,HNSC-TCGA-CV-7103-11A-01R-2015-13_mirnaB,BRCA-TCGA-BH-A1FB-11A-33R-A13P-13_mirnaA
anticodon,,,,,,,,,,,,,,,,,,,,,
AlaAGC,86950.921036,34892.666584,58926.334799,69168.922468,84822.552228,70837.402971,5.657801e+04,40140.821900,101665.286595,1.024850e+04,...,9.163716e+03,8.564120e+03,1.016988e+04,1.039961e+03,10166.099660,5.088347e+03,1.254285e+03,6484.967688,5207.435676,12589.054404
AlaCGC,7084.889862,2456.880506,3495.383181,5131.105814,5872.975921,2878.733357,4.043162e+03,3158.622051,4968.258349,1.389103e+03,...,9.087983e+02,5.561117e+02,1.242344e+03,5.045739e+02,960.576346,1.062402e+03,1.553695e+02,1045.237426,1015.634618,1578.691710
AlaGGC,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,32.902313,92.004784,1.543448e+01,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
AlaTGC,17969.857014,12532.572279,8330.663249,11393.479051,13088.346338,15627.409654,1.144687e+04,9245.549962,14444.751127,4.460565e+03,...,1.363197e+03,2.113224e+03,1.334797e+04,4.179104e+03,4122.473484,3.634534e+03,8.038752e+03,2101.713965,3280.807583,7569.624352
ArgGCG,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
iMetCAT,2189.875048,1166.397816,2184.614488,1939.315583,2600.889336,2364.673829,7.613747e+02,1283.190208,3036.157880,1.565056e+04,...,2.448075e+04,4.026248e+04,1.092107e+04,3.474242e+03,28737.242345,1.045627e+04,1.569879e+03,25018.263557,14951.372646,9836.463731
SeCTCA,579.672807,1141.580841,174.769159,646.438528,671.197248,771.089292,6.301032e+02,559.339322,1196.062195,1.528014e+03,...,1.590397e+03,1.112223e+03,2.195770e+03,2.233991e+02,760.456274,1.230150e+03,3.285417e+02,1787.018826,609.380771,1011.981865
UndetNNN,450.856628,818.960169,786.461216,444.426488,671.197248,565.465481,6.301032e+02,822.557826,736.038274,1.389103e+02,...,7.573319e+01,1.112223e+02,2.311337e+02,1.964372e+02,920.552331,2.236636e+02,5.502669e+01,550.716493,172.350117,121.437824


In [10]:
tRNA_abundancies

,COAD-TCGA-A6-2671-11A-01R-1757-13_mirna,COAD-TCGA-A6-2680-11A-01R-1757-13_mirna,COAD-TCGA-AA-3525-11A-01R-1757-13_mirna,READ-TCGA-AF-2689-11A-01R-1757-13_mirna,COAD-TCGA-A6-2685-11A-01R-1757-13_mirna,COAD-TCGA-A6-2683-11A-01R-1757-13_mirna,COAD-TCGA-AA-3527-11A-01R-1757-13_mirna,COAD-TCGA-A6-2684-11A-01R-1757-13_mirna,READ-TCGA-AF-2691-11A-01R-1757-13_mirna,GBM-TCGA-06-0675-11A-32R-A36C-13_mirna,...,BLCA-TCGA-BL-A13J-11A-13R-A10V-13_mirnaB,BRCA-TCGA-BH-A0E1-11A-13R-A090-13_mirnaA,UCEC-TCGA-DI-A2QU-11A-11R-A18L-13_mirnaB,STAD-TCGA-HU-A4GH-11A-11R-A360-13_mirna,PAAD-TCGA-HV-A5A3-11A-11R-A26Y-13_mirna,UCEC-TCGA-FL-A1YQ-11A-11R-A17A-13_mirnaA,LUSC-TCGA-56-7580-11A-01H-2044-13_mirnaA,HNSC-TCGA-CV-7103-11A-01R-2015-13_mirnaB,BRCA-TCGA-BH-A1FB-11A-33R-A13P-13_mirnaA,sum
anticodon,,,,,,,,,,,,,,,,,,,,,
AlaAGC,86950.921036,34892.666584,58926.334799,69168.922468,84822.552228,70837.402971,56578.014650,40140.821900,101665.286595,10248.495138,...,8564.119675,10169.883277,1039.961483,10166.099660,5088.347126,1254.284799,6484.967688,5207.435676,12589.054404,1.025961e+07
AlaCGC,7084.889862,2456.880506,3495.383181,5131.105814,5872.975921,2878.733357,4043.162068,3158.622051,4968.258349,1389.103257,...,556.111667,1242.343696,504.573905,960.576346,1062.402147,155.369472,1045.237426,1015.634618,1578.691710,1.275156e+06
AlaGGC,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,32.902313,92.004784,15.434481,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.277532e+03
AlaTGC,17969.857014,12532.572279,8330.663249,11393.479051,13088.346338,15627.409654,11446.874426,9245.549962,14444.751127,4460.564902,...,2113.224335,13347.971802,4179.104478,4122.473484,3634.533661,8038.751736,2101.713965,3280.807583,7569.624352,6.345070e+06
ArgGCG,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SupTTA,0.000000,74.450924,29.128193,0.000000,83.899656,0.000000,0.000000,164.511565,92.004784,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,12.310723,0.000000,3.784588e+03
iMetCAT,2189.875048,1166.397816,2184.614488,1939.315583,2600.889336,2364.673829,761.374675,1283.190208,3036.157880,15650.563359,...,40262.484707,10921.067838,3474.241695,28737.242345,10456.273764,1569.879038,25018.263557,14951.372646,9836.463731,1.690284e+07
SeCTCA,579.672807,1141.580841,174.769159,646.438528,671.197248,771.089292,630.103179,559.339322,1196.062195,1528.013582,...,1112.223334,2195.770253,223.399133,760.456274,1230.149855,328.541696,1787.018826,609.380771,1011.981865,1.784213e+06


In [30]:
# drop zero rows
tRNA_abundancies.loc[(tRNA_abundancies != 0).any(axis=1)].index[:-6]

# drop Selenocysteine
# tRNA_abundancies.drop(index='SeCTCA', inplace=True)

Index(['AlaAGC', 'AlaCGC', 'AlaGGC', 'AlaTGC', 'ArgACG', 'ArgCCG', 'ArgCCT',
       'ArgTCG', 'ArgTCT', 'AsnATT', 'AsnGTT', 'AspATC', 'AspGTC', 'CysACA',
       'CysGCA', 'GlnCTG', 'GlnTTG', 'GluCTC', 'GluTTC', 'GlyCCC', 'GlyGCC',
       'GlyTCC', 'HisGTG', 'IleAAT', 'IleGAT', 'IleTAT', 'LeuAAG', 'LeuCAA',
       'LeuCAG', 'LeuTAA', 'LeuTAG', 'LysCTT', 'LysTTT', 'MetCAT', 'PheGAA',
       'ProAGG', 'ProCGG', 'ProGGG', 'ProTGG', 'SerACT', 'SerAGA', 'SerCGA',
       'SerGCT', 'SerGGA', 'SerTGA', 'ThrAGT', 'ThrCGT', 'ThrTGT', 'TrpCCA',
       'TyrATA', 'TyrGTA', 'ValAAC', 'ValCAC', 'ValTAC'],
      dtype='object', name='anticodon')

In [25]:
# Remove READS, UndetNNN, SeCTCA, iMetCAT, SupTTA, SupTCA
60 - 1 - 1 - 1 - 1 - 1 - 1

54